### Dataset

In [33]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

class RxHandBDDataset(Dataset):
    def __init__(self, csv_path, image_dir, processor):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]  # Get the row at index idx

        image_path = os.path.join(self.image_dir, row["Images"])
        image = Image.open(image_path).convert("RGB")

        text = row["Text"]

        # Converting our image pixels to tensor
        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Convert our ground truth to token ids
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        # Dataloader automatically batches these so its fine
        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [35]:
dataset = RxHandBDDataset(
    "/workspace/rxhandbd/RxHandBD-ML/Train_Label.csv",
    "/workspace/rxhandbd/RxHandBD-ML/Train_Set",
    processor
)

train_dict = dataset[0]
pixel_values = train_dict["pixel_values"]
labels = train_dict["labels"]

print(pixel_values.shape)
print(labels.shape)
print(labels)

# Am I decoding correctly?
decode_labels = labels.clone()
decode_labels[decode_labels == -100] = processor.tokenizer.pad_token_id

text = processor.tokenizer.decode(
    decode_labels,
    skip_special_tokens=True
)

print(text)

torch.Size([3, 384, 384])
torch.Size([64])
tensor([    0, 46905,  1334,     2,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100])
Dexter


### DataLoader

In [41]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

# Testing one batch
# batch = next(iter(train_loader))

# pixel_values = batch["pixel_values"]
# labels = batch["labels"]

# print(pixel_values.shape)
# print(labels.shape)

# label = labels[3].clone()
# label[label == -100] = processor.tokenizer.pad_token_id

# print(processor.tokenizer.decode(label, skip_special_tokens=True))

### Align the Model to the Processor. 
Make sure that start, pad, and eos are the same

In [42]:
def align_model_to_processor(model, processor):
    tokenizer = processor.tokenizer

    model.config.decoder_start_token_id = tokenizer.cls_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.sep_token_id

    model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.sep_token_id

    # Useful for generation later
    model.generation_config.max_length = 64
    model.generation_config.num_beams = 1

    return model

### Define the VisionEncoderDecoder Model

In [47]:
import torch
from transformers import VisionEncoderDecoderModel

device = torch.device("cuda")

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-handwritten"
).to(device)

model = align_model_to_processor(model, processor)



Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Sanity check special token alignment

In [48]:
def check_trocr_alignment(model, processor):
    tokenizer = processor.tokenizer

    expected_decoder_start = tokenizer.cls_token_id
    expected_pad = tokenizer.pad_token_id
    expected_eos = tokenizer.sep_token_id

    checks = {
        "model.config.decoder_start_token_id": (
            model.config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.config.pad_token_id": (
            model.config.pad_token_id,
            expected_pad,
        ),
        "model.config.eos_token_id": (
            model.config.eos_token_id,
            expected_eos,
        ),
        "model.generation_config.decoder_start_token_id": (
            model.generation_config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.generation_config.pad_token_id": (
            model.generation_config.pad_token_id,
            expected_pad,
        ),
        "model.generation_config.eos_token_id": (
            model.generation_config.eos_token_id,
            expected_eos,
        ),
    }

    for name, (actual, expected) in checks.items():
        assert actual == expected, f"{name}: expected {expected}, got {actual}"

    print("TrOCR alignment check passed.")
    print(f"decoder_start_token_id = {expected_decoder_start}")
    print(f"pad_token_id           = {expected_pad}")
    print(f"eos_token_id           = {expected_eos}")

check_trocr_alignment(model, processor)

TrOCR alignment check passed.
decoder_start_token_id = 0
pad_token_id           = 1
eos_token_id           = 2


### Testing one batch

In [40]:
batch = next(iter(train_loader))

batch = {
    "pixel_values": batch["pixel_values"].to(device),
    "labels": batch["labels"].to(device),
}

outputs = model(**batch)

print(outputs.loss)

tensor(13.1837, device='cuda:0', grad_fn=<NllLossBackward0>)


### Testing a few batches

In [50]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 3


for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {
            "pixel_values": batch["pixel_values"].to(device),
            "labels": batch["labels"].to(device),
        }

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 25 == 0:
            print(f"epoch {epoch+1}, step {step}, loss = {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{num_epochs} finished | avg train loss = {avg_loss:.4f}")

epoch 1, step 0, loss = 4.9899
epoch 1, step 25, loss = 2.4659
epoch 1, step 50, loss = 1.7782
epoch 1, step 75, loss = 1.7360
epoch 1, step 100, loss = 2.2583
epoch 1, step 125, loss = 2.7080
epoch 1, step 150, loss = 1.4631
epoch 1, step 175, loss = 1.4097
epoch 1, step 200, loss = 3.0438
epoch 1, step 225, loss = 2.2720
epoch 1, step 250, loss = 1.3530
epoch 1, step 275, loss = 2.0713
epoch 1, step 300, loss = 2.2899
epoch 1, step 325, loss = 0.6460
epoch 1, step 350, loss = 0.7623
epoch 1, step 375, loss = 0.9016
epoch 1, step 400, loss = 0.6399
epoch 1, step 425, loss = 1.0247
epoch 1, step 450, loss = 0.2361
epoch 1, step 475, loss = 1.3229
epoch 1, step 500, loss = 1.4417
epoch 1, step 525, loss = 1.0380
epoch 1, step 550, loss = 1.2233
epoch 1, step 575, loss = 1.1723
epoch 1, step 600, loss = 0.8957
epoch 1, step 625, loss = 1.5953
epoch 1, step 650, loss = 2.6748
epoch 1, step 675, loss = 1.7611
epoch 1, step 700, loss = 0.9087
epoch 1, step 725, loss = 2.4626
epoch 1, step 7